# Tutorial 08 — Parameterized Inventories & Scenario Analysis

Companion explainer: **08_parameters_scenarios.md**. Two techniques:
(1) bw2data's stored parameter system (formulas travel with the model), and
(2) the plain-Python sweep pattern (build-function + loop) used in the case
studies. Plus a tornado chart.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc
from bw2data.parameters import parameters, ProjectParameter, ActivityParameter

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2 = find_flow("Carbon dioxide, fossil")
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))

## Part 1 — bw2data stored parameters + formulas

In [2]:
DB = "t08_param"
if DB in bd.databases:
    del bd.databases[DB]
bd.Database(DB).write({
    (DB, "elec"): {"name": "electricity", "unit": "kilowatt hour", "exchanges": [
        {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
        {"input": co2.key, "amount": 0.95, "type": "biosphere"}]},
    (DB, "steel"): {"name": "steel", "unit": "kilogram", "exchanges": [
        {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 2.9, "type": "technosphere"},
        {"input": co2.key, "amount": 1.9, "type": "biosphere"}]},
    (DB, "kettle"): {"name": "kettle", "unit": "unit", "exchanges": [
        {"input": (DB, "kettle"), "amount": 1.0, "type": "production"},
        {"input": (DB, "steel"), "amount": 1.2, "type": "technosphere"},
        {"input": (DB, "elec"), "amount": 0.8, "type": "technosphere"}]},
})
kettle = bd.get_node(database=DB, code="kettle")

# define a project parameter and drive the steel input by formula
parameters.new_project_parameters([{"name": "steel_kg", "amount": 1.2}], overwrite=True)
for exc in kettle.technosphere():
    if exc.input["name"] == "steel":
        exc["formula"] = "steel_kg"
        exc.save()
parameters.add_exchanges_to_group("kettle_grp", kettle)
ActivityParameter.recalculate_exchanges("kettle_grp")

def kettle_score():
    l = bc.LCA({kettle: 1}, method=gwp); l.lci(); l.lcia(); return l.score

print("steel_kg=1.2 -> GWP", round(kettle_score(), 3))

# change the parameter and cascade
for p in ProjectParameter.select().where(ProjectParameter.name == "steel_kg"):
    p.amount = 2.0; p.save()
parameters.recalculate()
new_amt = next(e for e in kettle.technosphere() if e.input["name"] == "steel")["amount"]
print("after recalc, steel exchange amount =", round(new_amt, 3))
print("steel_kg=2.0 -> GWP", round(kettle_score(), 3))

# reset
for p in ProjectParameter.select().where(ProjectParameter.name == "steel_kg"):
    p.amount = 1.2; p.save()
parameters.recalculate()

13:18:01-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

6.346

after recalc, steel exchange amount =

2.0

steel_kg=2.0 -> GWP

10.07

## Part 2 — the sweep pattern (build-function + loop)

In [3]:
SDB = "t08_sweep"

def build(grid_ci=0.95, steel_kg=1.2, steel_ci=1.9):
    if SDB in bd.databases:
        del bd.databases[SDB]
    bd.Database(SDB).write({
        (SDB, "elec"): {"name": "electricity", "unit": "kilowatt hour", "exchanges": [
            {"input": (SDB, "elec"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": grid_ci, "type": "biosphere"}]},
        (SDB, "steel"): {"name": "steel", "unit": "kilogram", "exchanges": [
            {"input": (SDB, "steel"), "amount": 1.0, "type": "production"},
            {"input": (SDB, "elec"), "amount": 2.9, "type": "technosphere"},
            {"input": co2.key, "amount": steel_ci, "type": "biosphere"}]},
        (SDB, "kettle"): {"name": "kettle", "unit": "unit", "exchanges": [
            {"input": (SDB, "kettle"), "amount": 1.0, "type": "production"},
            {"input": (SDB, "steel"), "amount": steel_kg, "type": "technosphere"},
            {"input": (SDB, "elec"), "amount": 0.8, "type": "technosphere"}]},
    })
    return bd.get_node(database=SDB, code="kettle")

def score(**kw):
    act = build(**kw)
    l = bc.LCA({act: 1}, method=gwp); l.lci(); l.lcia()
    return l.score

# grid decarbonization sweep
grid = np.linspace(0.1, 1.0, 10)
sweep = pd.DataFrame({"grid_ci": grid,
                      "GWP": [score(grid_ci=g) for g in grid]})
print(sweep.round(3).to_string(index=False))

13:18:03-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

 [

info     

] 

Vacuuming database            

 grid_ci   GWP
     0.1 2.708
     0.2 3.136
     0.3 3.564
     0.4 3.992
     0.5 4.420
     0.6 4.848
     0.7 5.276
     0.8 5.704
     0.9 6.132
     1.0 6.560

## Named scenarios table

In [4]:
base = dict(grid_ci=0.95, steel_kg=1.2, steel_ci=1.9)
SCENARIOS = {
    "baseline":    dict(base),
    "2030 grid":   {**base, "grid_ci": 0.45},
    "green steel": {**base, "grid_ci": 0.45, "steel_ci": 0.5},
    "lightweight": {**base, "steel_kg": 0.8},
}
scen = pd.DataFrame([{"scenario": k, "GWP": round(score(**v), 3)}
                     for k, v in SCENARIOS.items()])
print(scen.to_string(index=False))

13:18:07-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

 [

info     

] 

Vacuuming database            

   scenario   GWP
   baseline 6.346
  2030 grid 4.206
green steel 2.526
lightweight 4.484

## Tornado chart: one-at-a-time +-20%

In [5]:
base_score = score(**base)
tornado = []
for pname in ["grid_ci", "steel_kg", "steel_ci"]:
    lo = score(**{**base, pname: base[pname] * 0.8})
    hi = score(**{**base, pname: base[pname] * 1.2})
    tornado.append({"param": pname, "low": lo - base_score, "high": hi - base_score,
                    "swing": abs(hi - lo)})
tdf = pd.DataFrame(tornado).sort_values("swing")
print(tdf.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(tdf))
ax.barh(y, tdf["high"], color="#C44E52", label="+20%")
ax.barh(y, tdf["low"], color="#4C72B0", label="-20%")
ax.set_yticks(y); ax.set_yticklabels(tdf["param"])
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("change in GWP vs baseline (kg CO2-eq)")
ax.set_title(f"Tornado (baseline = {base_score:.2f})"); ax.legend()
plt.tight_layout()
plt.savefig("tutorials_outputs_08.png", dpi=120, bbox_inches="tight")
print("saved tutorials_outputs_08.png")
plt.show()

13:18:09-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

] 

Vacuuming database            

   param    low  high  swing
steel_ci -0.456 0.456  0.912
 grid_ci -0.813 0.813  1.626
steel_kg -1.117 1.117  2.234

saved tutorials_outputs_08.png

C:\Users\derne\AppData\Local\Temp\ipykernel_16612\1598528095.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Next: **09 — comparative LCA & visualization**.